# XOR Problem

In [ ]:
from neuron import *
from network import Network
import plotly.graph_objects as go
import plotly.express as px
import pandas as pd
from functools import partial

In [ ]:
def i_fn(t, i_max, t_start, t_end):
    if t < t_start or t > t_end:
        return 0
    return i_max

i_fn_map = {
    0: partial(i_fn, i_max=-5, t_start=0, t_end=10),
    1: partial(i_fn, i_max=5, t_start=0, t_end=10),
}

In [ ]:
x1 = 1
x2 = 0

network = Network([
    NeuronGroup(2, l=1000, d=10.0, r_a=25.0),
    Neuron(l=1000, d=10.0, r_a=25.0),
], g=0.4, e_syn=-65.0, delay=0.5, weight=1.0)
network.add_current_injector(i_fn_map[x1], delay=0.5)
network.add_current_injector(i_fn_map[x2], delay=0.5)
network.update_in_synapse(0, weight_pattern=lambda i, j: -1.0 if i == j else 1.0)
network.update_in_synapse(1, weight_pattern=lambda i, j: 1.0 if i == j else -1.0)
network.run(50)
print(network.out_spike)

In [ ]:
data = []
neurons = []
recorder = network.recoder
for neuron_group in network.layers:
    neurons.extend(neuron_group.neurons)
for i in range(len(neurons)):
    neuron = neurons[i]
    for t in range(0, recorder.n_t, 200):
        for x in range(neuron.n_x):
            data.append({"neuron": i, "time": t, "x": neuron.dx * x, "v": recorder.v[i][x, t]})
df = pd.DataFrame(data)

In [ ]:
fig = px.line(df, x="x", y="v", animation_frame="time", title="Voltage of Neurons", facet_row="neuron", range_y=[-100, 40])
fig.update_layout(height=1200)
fig.show()

In [ ]:
from plotly.subplots import make_subplots
import plotly.graph_objects as go

fig = make_subplots(rows=3, cols=1, subplot_titles=("Injector 1", "Injector 2", "Output Neuron"), x_title="Time (10ms)")
data1 = []
data2 = []
data3 = []
for t in range(0, 50, 1):
    data1.append(i_fn_map[x1](t))
    data2.append(i_fn_map[x2](t))
for t in range(0, 5000, 5):
    data3.append(recorder.v[2][0, t])
fig.add_trace(go.Scatter(x=list(range(0, 50, 1)), y=data1, name="Injector 1"), row=1, col=1)
fig.update_yaxes(row=1, col=1, title_text="Injected Current (mA)")
fig.add_trace(go.Scatter(x=list(range(0, 50, 1)), y=data2, name="Injector 2"), row=2, col=1)
fig.update_yaxes(row=2, col=1, title_text="Injected Current (mA)")
fig.add_trace(go.Scatter(x=list(range(0, 5000, 5)), y=data3, name="Output Neuron"), row=3, col=1)
fig.update_yaxes(range=[-100,40],row=3, col=1, title_text="Membrane Potential (mV)")
fig.update_layout(height=900, title_text="XOR Neural Network Simulation Results", showlegend=False)
fig.show()